# Exploring the Dataset: Intranet Apache Error Log

**Goal:** Understand the structure of `intranet_smith_russellmitchell_com-error_log.2` (Apache httpd error log) to design the `http_errors` table.

This notebook walks through:
1. Loading the raw error log (37 lines of `[timestamp] [module:level] [pid] [client] message` format)
2. Parsing the Apache error log format (bracket-delimited fields + free-text message body)
3. Exploring every field at every nesting level
4. Building a raw 1:1 DataFrame
5. Identifying attack-relevant patterns (reconnaissance, forbidden probes, wp-* scanning)
6. Mapping to planned SQL schema with both PostgreSQL and MySQL types
7. Checking for 1NF, 2NF, and 3NF violations

---

**Dataset:** AIT Log Data Set V2.0 -- russellmitchell testbed  
**Source:** https://zenodo.org/records/5789064  
**Host:** intranet_server (main attack target, WordPress/intranet)  
**Linear issue:** DAT-43

## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
  data-201-security-log-analysis/   <-- this repo
    notebooks/                       <-- this notebook is here
  russellmitchell/                   <-- dataset is here
```

In [ ]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"

ERROR_LOG = (
    DATASET_ROOT
    / "gather"
    / "intranet_server"
    / "logs"
    / "apache2"
    / "intranet_smith_russellmitchell_com-error_log.2"
)

for p, name in [
    (DATASET_ROOT, "Dataset root"),
    (ERROR_LOG, "Error log"),
]:
    status = "FOUND" if p.exists() else "MISSING"
    print(f"{name}: {p.resolve()} [{status}]")

## 1. Load Raw Data

The Apache error log uses a structured bracket-delimited format. Each line has the structure:
```
[day Mon DD HH:MM:SS.usec YYYY] [module:level] [pid N] [client IP:port] message
```

Two nesting levels:
- **Outer level:** timestamp, module, severity level, pid, client IP and port
- **Message body:** Free-text description that varies by module — contains Apache error codes (AH#####), file paths, and occasionally a `referer:` suffix

In [ ]:
# Load all lines with 1-based line numbers
with open(ERROR_LOG) as f:
    raw_lines = f.readlines()

print(f"Total lines: {len(raw_lines)}")
print()
print("First 5 lines:")
for i, line in enumerate(raw_lines[:5], 1):
    print(f"  [{i}] {line.rstrip()}")
print()
print("Last 3 lines:")
for i, line in enumerate(raw_lines[-3:], len(raw_lines) - 2):
    print(f"  [{i}] {line.rstrip()}")

## 2. Parse the Apache Error Log Format

### 2.1 Module/level inventory

First, extract all distinct `[module:level]` combinations and their counts.

In [ ]:
import re
from collections import Counter

# Pattern: [timestamp] [module:level] [pid N] [client IP:port] message
LOG_RE = re.compile(
    r"^\[(?P<timestamp>[^\]]+)\]\s+"
    r"\[(?P<module>[^:]+):(?P<level>[^\]]+)\]\s+"
    r"\[pid\s+(?P<pid>\d+)\]\s+"
    r"(?:\[client\s+(?P<client_ip>[^:]+):(?P<client_port>\d+)\]\s+)?"
    r"(?P<message>.+)$"
)

module_level_counts = Counter()
for line in raw_lines:
    m = LOG_RE.match(line.rstrip())
    if m:
        module_level_counts[f"{m.group('module')}:{m.group('level')}"] += 1

print(f"Distinct module:level combinations: {len(module_level_counts)}")
print()
for key, count in module_level_counts.most_common():
    print(f"  {key:30s} {count:5d} ({count / len(raw_lines) * 100:5.1f}%)")

### 2.2 Format categories

The Apache error log entries fall into distinct categories based on their module and message pattern.
Let's examine a sample from each category.

In [ ]:
# Group lines by module and show one example of each
module_examples = {}
for line in raw_lines:
    m = LOG_RE.match(line.rstrip())
    if m and m.group("module") not in module_examples:
        module_examples[m.group("module")] = line.rstrip()

for module, example in sorted(module_examples.items()):
    print(f"--- {module} ---")
    print(f"  {example}")
    print()

### 2.3 Apache error code inventory

Apache error messages often begin with a code in the format `AH#####`. Extract all distinct codes.

In [ ]:
error_code_counts = Counter()
for line in raw_lines:
    codes = re.findall(r"AH\d{5}", line)
    for code in codes:
        error_code_counts[code] += 1

print(f"Distinct Apache error codes: {len(error_code_counts)}")
print()
for code, count in error_code_counts.most_common():
    print(f"  {code}  {count:5d} ({count / len(raw_lines) * 100:5.1f}%)")

# Lines with no error code
no_code = sum(1 for line in raw_lines if not re.search(r"AH\d{5}", line))
print()
print(f"Lines without an AH##### code: {no_code} ({no_code / len(raw_lines) * 100:.1f}%)")

### 2.4 Full parse into a flat dictionary per line

Parse all fields including optional `referer:` suffix in the message body.

In [ ]:
import pandas as pd

# Regex for optional referer at end of message
REFERER_RE = re.compile(r",?\s*referer:\s*(\S+)$")
# Regex for AH error code at start of message
AHCODE_RE = re.compile(r"^(AH\d{5}):\s*(.*)")
# Regex for file path in message
PATH_RE = re.compile(r'(?:script|file|directory)\s+[\'"]?(/[^\s\'"]+)[\'"]?')

parsed = []
for line_number, line in enumerate(raw_lines, 1):
    line = line.rstrip()
    m = LOG_RE.match(line)
    if not m:
        continue

    record = {
        "line_number": line_number,
        "raw_timestamp": m.group("timestamp"),
        "module": m.group("module"),
        "level": m.group("level"),
        "pid": m.group("pid"),
        "client_ip": m.group("client_ip"),
        "client_port": m.group("client_port"),
        "message_raw": m.group("message"),
    }

    # Strip referer from message
    msg = m.group("message")
    ref_m = REFERER_RE.search(msg)
    if ref_m:
        record["referer"] = ref_m.group(1)
        msg = msg[: ref_m.start()].rstrip()
    else:
        record["referer"] = None

    # Extract AH error code
    ah_m = AHCODE_RE.match(msg)
    if ah_m:
        record["error_code"] = ah_m.group(1)
        record["message"] = ah_m.group(2)
    else:
        record["error_code"] = None
        record["message"] = msg

    # Extract file/directory path from message
    path_m = PATH_RE.search(record["message"])
    record["target_path"] = path_m.group(1) if path_m else None

    # Parse timestamp  →  e.g. "Mon Jan 24 03:57:26.696483 2022"
    try:
        record["timestamp"] = pd.to_datetime(
            m.group("timestamp"), format="%a %b %d %H:%M:%S.%f %Y", utc=True
        )
    except Exception:
        record["timestamp"] = None

    parsed.append(record)

print(f"Parsed records: {len(parsed)} / {len(raw_lines)} lines")

## 3. Build the Raw DataFrame

In [ ]:
df = pd.DataFrame(parsed)

# Cast numeric columns
df["pid"] = pd.to_numeric(df["pid"], errors="coerce").astype("Int64")
df["client_port"] = pd.to_numeric(df["client_port"], errors="coerce").astype("Int64")

print(f"Shape: {df.shape}")
print(f"Columns ({len(df.columns)}):")
for col in df.columns:
    non_null = df[col].notna().sum()
    nunique = df[col].nunique()
    dtype = df[col].dtype
    print(f"  {col:25s}  non-null: {non_null:4d}/{len(df)}  unique: {nunique:4d}  dtype: {dtype}")

## 4. Field-by-Field Exploration

### 4.1 module and level

In [ ]:
print("=== module distribution ===")
module_dist = df["module"].value_counts()
for mod, count in module_dist.items():
    print(f"  {mod:20s} {count:5d} ({count / len(df) * 100:5.1f}%)")

print()
print("=== level distribution ===")
level_dist = df["level"].value_counts()
for lvl, count in level_dist.items():
    print(f"  {lvl:20s} {count:5d} ({count / len(df) * 100:5.1f}%)")

### 4.2 Timestamp range

In [ ]:
print("=== Timestamp range ===")
print(f"  Earliest: {df['timestamp'].min()}")
print(f"  Latest:   {df['timestamp'].max()}")
print(f"  Span:     {df['timestamp'].max() - df['timestamp'].min()}")

print()
print("=== Events per second (burst analysis) ===")
# All events in this log happen in a tight window; show unique seconds
seconds = df["timestamp"].dt.floor("s")
burst = seconds.value_counts().sort_index()
for ts, count in burst.items():
    print(f"  {ts}  {count:3d} events")

### 4.3 client_ip and client_port

In [ ]:
for col in ["client_ip", "client_port"]:
    print(f"=== {col} ===")
    non_null = df[col].notna().sum()
    print(f"  Present: {non_null}/{len(df)}")
    if non_null > 0:
        vals = df[col].dropna().value_counts()
        print(f"  Unique values: {len(vals)}")
        if len(vals) <= 20:
            for v, c in vals.items():
                print(f"    {v}: {c}")
        else:
            print("  Top 10:")
            for v, c in vals.head(10).items():
                print(f"    {v}: {c}")
    print()

### 4.4 error_code

In [ ]:
print("=== error_code ===")
non_null = df["error_code"].notna().sum()
print(f"  Present: {non_null}/{len(df)}")
print()

# Map known AH codes to human meanings
AH_MEANINGS = {
    "AH01630": "client denied by server configuration (authz_core)",
    "AH00687": "content negotiation: file(s) found but none could be negotiated",
    "AH01276": "cannot serve directory: no DirectoryIndex found and autoindex forbidden",
}

code_dist = df["error_code"].value_counts(dropna=False)
for code, count in code_dist.items():
    meaning = (
        AH_MEANINGS.get(str(code), "(no code / php7 script-not-found)")
        if pd.notna(code)
        else "(no code / php7 script-not-found)"
    )
    print(f"  {str(code):10s}  {count:4d} ({count / len(df) * 100:5.1f}%)  {meaning}")

### 4.5 target_path (extracted file/directory from message body)

In [ ]:
print("=== target_path ===")
non_null = df["target_path"].notna().sum()
print(f"  Present: {non_null}/{len(df)}")
print()

path_dist = df["target_path"].value_counts()
print("  All target paths:")
for path, count in path_dist.items():
    print(f"  {count:3d}x  {path}")

### 4.6 referer

In [ ]:
print("=== referer ===")
non_null = df["referer"].notna().sum()
print(f"  Present: {non_null}/{len(df)} ({non_null / len(df) * 100:.1f}%)")
print()

ref_dist = df["referer"].value_counts()
for ref, count in ref_dist.items():
    print(f"  {count:3d}x  {ref}")

print()
print("Lines WITHOUT a referer (early probe phase):")
no_ref = df[df["referer"].isna()]
for _, row in no_ref.iterrows():
    print(f"  line {row['line_number']:3d}: [{row['module']}] {row['message_raw'][:90]}")

### 4.7 pid

In [ ]:
print("=== pid ===")
non_null = df["pid"].notna().sum()
print(f"  Present: {non_null}/{len(df)}")
pid_vals = df["pid"].value_counts()
print(f"  Unique values: {len(pid_vals)}")
print("  Top 10:")
for pid, count in pid_vals.head(10).items():
    print(f"    {pid}: {count}")

## 5. Attack Pattern Analysis

### 5.1 Reconnaissance timeline

All 37 errors originate from a single source IP in a ~2-minute burst. Map the sequence of probes.

In [ ]:
print("=== Full attack timeline (chronological) ===")
print()
for _, row in df.sort_values("timestamp").iterrows():
    ts = row["timestamp"].strftime("%H:%M:%S") if pd.notna(row["timestamp"]) else "??:??:??"
    mod = row["module"]
    msg = row["message"][:70]
    ref = " [referer]" if pd.notna(row["referer"]) else ""
    print(f"  {ts}  [{mod:12s}]  {msg}{ref}")

### 5.2 WordPress-specific probe classification

Classify each request by the type of WordPress vulnerability being probed.

In [ ]:
WP_CATEGORIES = {
    "htaccess_probe": [r"\.hta"],
    "admin_probe": [r"admin\.php"],
    "info_probe": [r"info\.php", r"phpinfo\.php"],
    "server_status": [r"server-status"],
    "wp_core_probe": [
        r"wp-(?:blog-header|config|cron|links-opml|load|login|mail|settings|signup|trackback)"
    ],
    "xmlrpc_probe": [r"xmlrpc"],
    "db_tool_probe": [r"searchreplacedb2\.php"],
    "uploads_browse": [r"wp-content/uploads"],
    "emergency_shell": [r"emergency\.php"],
    "php_fatal": [r"Call to undefined function"],
    "timthumb_probe": [r"timthumb", r"thumb\.php"],
    "functions_probe": [r"functions"],
}


def classify_probe(row):
    text = row["message_raw"] or ""
    for cat, patterns in WP_CATEGORIES.items():
        for pat in patterns:
            if re.search(pat, text, re.IGNORECASE):
                return cat
    return "other"


df["probe_category"] = df.apply(classify_probe, axis=1)

print("=== Probe category counts ===")
for cat, count in df["probe_category"].value_counts().items():
    print(f"  {cat:25s} {count:4d} ({count / len(df) * 100:5.1f}%)")

### 5.3 Probe phase vs. exploitation phase

The referer field distinguishes the two phases:
- **No referer:** Initial blind scanning (htaccess, admin, info, server-status, wp-core, xmlrpc)
- **Referer present:** Second-stage targeted probes after the attacker browsed the site

In [ ]:
phase1 = df[df["referer"].isna()]
phase2 = df[df["referer"].notna()]

print(f"Phase 1 (no referer, blind scan):   {len(phase1):3d} events")
print(f"Phase 2 (referer present, targeted): {len(phase2):3d} events")
print()

print("Phase 1 probe categories:")
for cat, count in phase1["probe_category"].value_counts().items():
    print(f"  {cat:25s} {count}")

print()
print("Phase 2 probe categories:")
for cat, count in phase2["probe_category"].value_counts().items():
    print(f"  {cat:25s} {count}")

## 6. Cross-Log Correlation

### 6.1 IP overlap with audit log

The attacker IP `172.19.131.174` appears in both this error log and in the audit log (USER_LOGIN events). Confirm the overlap and timestamp relationship.

In [ ]:
ATTACKER_IP = "172.19.131.174"

attacker_rows = df[df["client_ip"] == ATTACKER_IP]
print(f"Rows from attacker IP ({ATTACKER_IP}): {len(attacker_rows)} / {len(df)}")
print()
print("Error log window:")
print(f"  Earliest: {attacker_rows['timestamp'].min()}")
print(f"  Latest:   {attacker_rows['timestamp'].max()}")
print()
print("Note: In the audit log (DAT-42), this IP appears in USER_LOGIN events at ~2022-01-21")
print("  and in USER_AUTH/SYSCALL privilege escalation events at ~2022-01-24.")
print("  The error log timestamps (2022-01-24 03:57-03:58 UTC) fall within the attack window.")

## 7. SQL Schema Design

### 7.1 Planned `http_errors` table (PostgreSQL and MySQL DDL)

In [ ]:
postgresql_ddl = """
-- PostgreSQL DDL for http_errors (raw 1:1 with error log lines)
CREATE TABLE http_errors (
    row_id          SERIAL PRIMARY KEY,
    line_number     INT             NOT NULL,
    timestamp       TIMESTAMPTZ,
    raw_timestamp   VARCHAR(40),
    module          VARCHAR(20)     NOT NULL,
    level           VARCHAR(10)     NOT NULL,
    pid             INT,
    client_ip       INET,
    client_port     INT,
    error_code      VARCHAR(10),
    message         TEXT,
    message_raw     TEXT,
    target_path     TEXT,
    referer         TEXT
);
"""

mysql_ddl = """
-- MySQL DDL for http_errors (raw 1:1 with error log lines)
CREATE TABLE http_errors (
    row_id          INT AUTO_INCREMENT PRIMARY KEY,
    line_number     INT             NOT NULL,
    `timestamp`     DATETIME(6),
    raw_timestamp   VARCHAR(40),
    module          VARCHAR(20)     NOT NULL,
    level           VARCHAR(10)     NOT NULL,
    pid             INT,
    client_ip       VARCHAR(45),
    client_port     INT,
    error_code      VARCHAR(10),
    message         TEXT,
    message_raw     TEXT,
    target_path     TEXT,
    referer         TEXT
);
"""

print(postgresql_ddl)
print(mysql_ddl)

## 8. Normalization Observations

Applying the `normalization_rules_sheet.md` checklist to the raw Apache error log data.

### 8.1 1NF Check

**Multi-valued field:** The `message_raw` column packs all human-readable detail into a single text blob, including the target file path, error context, and sometimes a referer URL. This mirrors the `msg` blob in the audit log (DAT-42). Unpacking target_path and referer into separate columns (as done here) partially resolves the violation, but the `message` column still contains freeform text that cannot be atomized without type-specific parsing.

**Repeating groups:** None. No field1, field2, field3 patterns.

**1NF status: partially violated.** `message_raw` packs multiple values. The parsed `target_path` and `referer` extractions are 1NF remediation steps.

### 8.2 2NF Check

**Primary key:** Single-column surrogate (`row_id`). Partial dependencies require a composite PK, which does not exist here.

**2NF status: satisfied.** Single-column PK makes partial dependencies impossible by definition.

### 8.3 3NF Check

**Transitive dependencies identified:**

| Determinant | Dependent(s) | Pattern | Notes |
|-------------|-------------|---------|-------|
| module | level, error_code pattern | Each Apache module produces a characteristic severity level and error code pattern. `authz_core` always produces AH01630 at `error` level; `php7` produces no AH code; `negotiation` produces AH00687. | Structural FD: module → {level, error_code_pattern}. Drives 3NF decomposition to a module-definition lookup table. |
| error_code | message template | AH01630 always means "client denied by server configuration"; AH00687 always means negotiation failure. error_code → message_prefix. | The specific file path varies, but the error description prefix is determined entirely by error_code. |

**3NF status: violated.** The module → error_code and error_code → message_template chains are transitive dependencies through non-key attributes.

### 8.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|---|---|---|---|
| FD1 | row_id | all attributes | Surrogate PK, trivially determines everything. |
| FD2 | line_number | all attributes | Each line number uniquely identifies one log entry within this file. Candidate key (within this file; across files, needs source_file). |
| FD3 | error_code | message_prefix | The AH##### code determines the human-readable error description prefix. The path suffix varies per event. |
| FD4 | module | {level, error_code_range} | Structural FD: module determines which severity levels and error codes are possible. |
| FD5 | client_ip | probe_intent | In this dataset, the single source IP 172.19.131.174 is the known attacker, making client_ip a determinant of attack classification. (Dataset-specific, not a general schema FD.) |

## 9. Key Findings for Schema Design

1. **Single-source attack burst:** All 37 errors originate from one IP (`172.19.131.174`) in a ~2-minute window (03:57–03:58 UTC, 2022-01-24). This is a textbook automated reconnaissance sweep, not organic user traffic. The log contains no baseline "normal" HTTP error activity to distinguish from.

2. **Two-phase attack structure:** Lines 1–11 (no referer) are blind initial probes — htaccess files, admin.php, phpinfo.php, server-status, wp-core files, xmlrpc. Lines 12–37 (referer present) are second-stage targeted scans after the attacker navigated the site, focusing on timthumb.php variants and WordPress internals.

3. **Module heterogeneity across three modules:** `authz_core` (6 lines, AH01630: access denied), `php7` (19 lines, no AH code: script not found or PHP fatal), `negotiation` (11 lines, AH00687: content negotiation failure), `autoindex` (1 line, AH01276: directory listing forbidden). The module determines the error_code pattern — a 3NF transitive dependency.

4. **1NF violation in message column:** The raw message packs target path, error context, and referer into one text blob. Partially resolved by extracting `target_path` and `referer` as separate columns. The remaining `message` TEXT field still violates 1NF for complex entries.

5. **PHP Fatal error is the most actionable line:** Line 27 (03:57:56) contains a PHP Fatal error — `Call to undefined function get_header()` in `wp-content/themes/go/index.php:15`. This indicates the attacker successfully triggered WordPress theme code, confirming the server is running WordPress with the "Go" theme. Unlike the other 36 lines (probes returning 403/404), this is server-side code execution.

6. **timthumb.php is a high-signal probe:** Lines 28–36 probe 7+ timthumb.php locations across themes and wp-content. timthumb is a known vulnerable image resizing script (CVE-2011-4106). The attacker systematically checks every common install location. All return 404 (script not found).

7. **No authentication or session data:** Unlike the audit log (DAT-42), the Apache error log contains no uid, auid, or session information. The only identity signal is client_ip. Correlation with audit log USER_LOGIN events (same IP, same date) is required to associate these HTTP probes with the jhall account used in the privilege escalation attack.

8. **Merge with access log (future DAT):** The error log only captures failed requests. The companion access log would show successful HTTP requests from the same IP in the same window, including any 200 OK responses that indicate successful page enumeration. The `http_errors` table should include a `host_id` FK for multi-host joins.

9. **client_port is unique per TCP connection:** 26 distinct client ports across 37 entries, meaning the attacker used persistent connections (multiple errors per port) for some probes. Port reuse pattern: ports 36072, 36076, 36110, 36128, etc. follow a sequential allocation pattern consistent with a single scanning process.